# LODissea project
## A journey through European(a) cultural heritage database

### Abstract 
Europeana is really the place to discover Europe's digital cultural heritage? This exploratory research investigates whether Europeana functions effectively as a collaborative shared space for digital cultural heritage across Europe. Specifically, we examine the scale and distribution of active data providers, analyzing who they are, how they are distributed geographically and in percentage terms, what institutional categories they represent, and the overall structural quality of their shared data. The selected country for this exploration are: Italy, Germany, Spain, Portugal, France, Netherdlands. 

#### Preliminary: Shared europeana key
The Europeana API key is loaded from a `.env` file (`EUROPEANA_API_KEY=...`). Get a free key at https://api.europeana.eu/.

In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv

# API KEY: reads EUROPEANA_API_KEY from a local .env file
load_dotenv()
EUROPEANA_API_KEY = os.environ.get("EUROPEANA_API_KEY", "")
if not EUROPEANA_API_KEY:
    print("WARNING: EUROPEANA_API_KEY not set.\n"
          "Create a .env file with: EUROPEANA_API_KEY=your_key_here\n"
          "The notebook will fall back to cached CSV/JSON files where available.")

EUROPEANA_SEARCH_URL = "https://api.europeana.eu/record/v2/search.json"
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
(DATA_DIR / "csv").mkdir(exist_ok=True)
(DATA_DIR / "json").mkdir(exist_ok=True)
REQUEST_DELAY = 0.3


#### Preliminary: Shared color palette
All country-level visualizations use this single color mapping for consistency.

In [5]:
# Country colors (reused across all visualizations)
COUNTRY_COLORS = {
    "netherlands": "#a180ad",
    "france":      "#1f7f95",
    "portugal":    "#f4a64e",
    "italy":       "#90BE6D",
    "germany":     "#feda15",
    "spain":       "#bb521f",
}

CATEGORY_COLORS = {
    "audiovisual/film archive": "#80CBC4",
    "art/history museum": "#9FA8DA",
    "natural history/science institution": "#CE93D8",
    "library/archive": "#90CAF9",
    "academic/research institution": "#FFCC80",
    "media/broadcast organization": "#EF9A9A",
    "government/administrative body": "#E8A0BE",
    "other": "#CFCFCF",
    "unresolved": "#9E9E9E",
}

# ISO-code variant for RQ1 visualizations (uses iso_code column)
ISO_TO_LOWER = {
    "IT": "italy", "DE": "germany", "NL": "netherlands",
    "PT": "portugal", "ES": "spain", "FR": "france",
}
COUNTRY_COLORS_ISO = {iso: COUNTRY_COLORS[lower] for iso, lower in ISO_TO_LOWER.items()}

print("Country color palette:")
for name, color in COUNTRY_COLORS.items():
    print(f"  {name:<15} {color}")


Country color palette:
  netherlands     #a180ad
  france          #1f7f95
  portugal        #f4a64e
  italy           #90BE6D
  germany         #feda15
  spain           #bb521f


## RQ01 Does the number of digital heritage and providers in Europeana reflect GLAM density or cultural expenditure (% GDP)?
### RsQ01 The number of Europeana providers reflect the number of GLAM institutions?
We began our inquiry by asking whether the volume of cultural heritage shared on Europeana reflects the true distribution of physical GLAM institutions across each country. Through this exploratory sub-question, we sought to understand whether Europeana functions as an equitable digital representation of national cultural heritage, and whether the number of active data providers scales proportionally with the underlying physical infrastructure. We interrogated Europeana data to find the number of providers for each country and Wikidata to find the number of GLAM institution.

### RsQ02 The volume of cultural heritage items shared via Europeana is proportional to the investment in culture done by each country?
Our second sub-question investigate further the relationship between investement and Europeana precence. With this exploration we are trying to understand if the selected country invest in sharing their digitaized cultural heritage on the European(a) platform. The data to answer this question was obtain from Eurostat and Europeana. 

### RsQ03 Does the number of active provider and volume of cultural heritage shared on Europeana correlate positevely to cultural investment? 
The third sub-question try to provide a summative exploration by observing the correlation between investments, Europeana providers to GLAM institutions rate and volume of Europeana data. 

#### Install dependencies and import

In [6]:
# Install libraries

!pip install SPARQLWrapper rdflib eurostat plotly seaborn

In [7]:
# Import libraries
import requests
import json
import time
import pandas as pd
import eurostat
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from SPARQLWrapper import SPARQLWrapper, JSON

#### Mapping each country 
We mapped each country to a standardized dictionary of ISO codes, English names, and Wikidata QIDs was essential to establish a common relational key across four distinct data ecosystems (Eurostat, Europeana, Wikidata, and national censuses).

In [8]:
# Mapping countries
countries = {
    'IT': {'name_en': 'Italy','wd_id': 'Q38'},
    'DE': {'name_en': 'Germany', 'wd_id': 'Q183'},
    'NL': {'name_en': 'Netherlands', 'wd_id': 'Q55'},
    'PT': {'name_en': 'Portugal', 'wd_id': 'Q45'},
    'ES': {'name_en': 'Spain', 'wd_id': 'Q29'},
    'FR': {'name_en': 'France','wd_id': 'Q142'}
}

df_countries = pd.DataFrame.from_dict(countries, orient='index').reset_index()
df_countries.rename(columns={'index': 'iso_code'}, inplace=True)
df_countries

,iso_code,name_en,wd_id
0,IT,Italy,Q38
1,DE,Germany,Q183
2,NL,Netherlands,Q55
3,PT,Portugal,Q45
4,ES,Spain,Q29
5,FR,France,Q142


#### Querying the Europeana API for n° of cultural heritage items and n° of providers for each country
Data is collected via the Europeana Search REST API (`/record/v2/search.json`). Two sequential queries are executed per ISO 3166-1 alpha-2 country code:
* **Item Volume Extraction**: Sets parameter `rows=0` to retrieve the aggregate `totalResults` count per country.
* **Provider Facet Extraction**: Uses `profile='facets'`, `facet='DATA_PROVIDER'`, and `f.DATA_PROVIDER.facet.limit=1500` to extract unique contributing institutions without downloading record-level metadata.

*Note: A 0.5-second rate-limiting delay (`time.sleep`) is implemented to adhere to API request thresholds. Error handling defaults missing or failed requests to `0`.*


*Note: The europeana total items per country count accounts for Tier 0 entries as well, despite them not being visible on the website.*

In [9]:
# Europeana Data & Providers
API_KEY = EUROPEANA_API_KEY

europeana_data = []

for iso, info in countries.items():
    url = "https://api.europeana.eu/record/v2/search.json"
    params = {
        "wskey": API_KEY,
        "query": "*",
        "qf": f"COUNTRY:{info["name_en"].lower()}",
        "rows": 0,
        "profile": "facets",
        "facet": "DATA_PROVIDER",
        "f.DATA_PROVIDER.facet.limit": 1500
    }

    try:
        response = requests.get(url, params=params, timeout=20)

        if response.status_code == 200:
            resp_json = response.json()
            totale = resp_json.get("totalResults", 0)
            facets = resp_json.get("facets", [])

            provider_count = 0
            for facet in facets:
                if facet.get("name") == "DATA_PROVIDER":
                    provider_count = len(facet.get("fields", []))
                    break

            europeana_data.append({"iso_code": iso, "europeana_total": totale, "europeana_providers": provider_count})

        else:
            errore_msg = response.json().get("error", response.text)
            print(f"   Error HTTP {response.status_code}: {errore_msg}")
            europeana_data.append({"iso_code": iso, "europeana_total": 0, "europeana_providers": 0})

    except Exception as e:
        print(f"   Connection error for {info["name_en"]}: {e}")
        europeana_data.append({"iso_code": iso, "europeana_total": 0, "europeana_providers": 0})

    time.sleep(0.5)

df_europeana = pd.DataFrame(europeana_data).sort_values(by="europeana_total", ascending=False, ignore_index=True)
df_europeana


,iso_code,europeana_total,europeana_providers
0,NL,9204845,104
1,DE,8701240,375
2,ES,6581724,275
3,FR,4724898,50
4,IT,1832376,158
5,PT,139858,40


#### Querying the Eurostat database for expenditure for culture
To obtain macroeconomic spending metrics, the remote Eurostat database is queried directly into a Pandas DataFrame using the eurostat Python library.

* **Dataset Ingested**: General Government expenditure by function (`gov_10a_exp`).

* **Filtering Criteria**: Data is filtered to isolate General Government sector expenditure (`sector = 'S13'`) within the Culture function (`cofog99 = 'GF08'`), expressed as a percentage of national GDP (`'unit = 'PC_GDP'`).

* **Output**: Returns a filtered DataFrame containing the 2022 expenditure percentages for the selected Member States, which are mapped to ISO alpha-2 country codes under the column culture_expenditure_gdp_2022


In [10]:
# Public expenditure for culture (% of GDP)
from pathlib import Path
import os

csv_path = Path("data/csv/eurostat_culture_expenditure.csv")
if csv_path.exists():
    print("Loading Eurostat data from cache...")
    df_exp_filtered = pd.read_csv(csv_path)
else:
    print("Fetching Eurostat data...")
    df_exp = eurostat.get_data_df("gov_10a_exp")

    df_exp_filtered = df_exp[
        (df_exp["sector"] == "S13") & #General government
        (df_exp["unit"] == "PC_GDP") & #Percentage of GDP
        (df_exp["cofog99"] == "GF08") & #Culture sector
        (df_exp["na_item"] == "TE") &
        (df_exp["geo\\TIME_PERIOD"].isin(countries.keys()))
    ][["geo\\TIME_PERIOD", "2022"]].rename(columns={"geo\\TIME_PERIOD": "iso_code", "2022": "culture_expenditure_gpd_2022"})
    
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df_exp_filtered.to_csv(csv_path, index=False)

df_exp_filtered


Loading Eurostat data from cache...


,iso_code,culture_expenditure_gpd_2022
0,DE,1.0
1,ES,1.2
2,FR,1.4
3,IT,0.9
4,NL,1.1
5,PT,0.9


#### Querying the remote Wikidata SPARQL endpoint for count of GLAM institutions
To establish a baseline count of physical GLAM institutions, the Wikidata Query Service endpoint (`https://query.wikidata.org/sparql`) is queried using `SPARQLWrapper`. 

* **Entity Classes Queried**: Museums (`wd:Q33506`), Libraries (`wd:Q7075`), Archives (`wd:Q166118`), and Art Galleries (`wd:Q1007870`).
* **Batch Processing**: Country QIDs are batched in groups of 3 (`BATCH_SIZE = 3`) using the `wdt:P17` (country) property to optimize query execution and prevent endpoint timeouts.
* **Output**: Returns distinct entity counts grouped by country QID, which are mapped back to ISO alpha-2 codes.

In [11]:
# Wikidata GLAM institutes for countries
from pathlib import Path
import os

csv_path = Path("data/csv/wikidata_glam_institutes.csv")
if csv_path.exists():
    print("Loading Wikidata GLAM data from cache...")
    df_wikidata = pd.read_csv(csv_path)
else:
    print("Fetching Wikidata GLAM data...")
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
    sparql.agent = "info-viz-student-project/1.0 (mailto:nome.cognome@studio.unibo.it)"
    sparql.setReturnFormat(JSON)
    
    countries_list = [(iso, info['wd_id'], info['name_en']) for iso, info in countries.items()]
    
    BATCH_SIZE = 3
    wikidata_rows = []
    
    def create_batch(list, dimension):
        for i in range(0, len(list), dimension):
            yield list[i:i + dimension]
    
    for index_batch, batch in enumerate(create_batch(countries_list, BATCH_SIZE), start=1):
        nomi_lotto = ", ".join([p[2] for p in batch])
    
        qid_clean = []
        for p in batch:
            raw_qid = p[1]
            clean_qid = raw_qid.split("/")[-1].replace("wd:", "")
            qid_clean.append(f"wd:{clean_qid}")
    
        string_values = " ".join(qid_clean)
    
        # museums, libraries, archives, galleries
        query_batch = f"""
        SELECT ?country (COUNT(DISTINCT ?item) AS ?count) WHERE {{
          VALUES ?country {{ {string_values} }}
          VALUES ?type {{ wd:Q33506 wd:Q7075 wd:Q166118 wd:Q1007870 }}
    
          ?item wdt:P31 ?type ;
                wdt:P17 ?country .
        }}
        GROUP BY ?country
        """
    
        sparql.setQuery(query_batch)
    
        try:
            results = sparql.query().convert()
    
            count_temp = {}
            for row in results["results"]["bindings"]:
                qid = row["country"]["value"].split("/")[-1]
                count_temp[qid] = int(row["count"]["value"])
    
            for iso, qid, nome in batch:
                clean_qid = qid.split("/")[-1].replace("wd:", "")
                values = count_temp.get(clean_qid, 0)
                wikidata_rows.append({'iso_code': iso, 'glam_count_wikidata': values})
    
        except Exception as e:
            print(f"   Batch Error {index_batch}: {e}")
            for iso, qid, nome in batch:
                wikidata_rows.append({'iso_code': iso, 'glam_count_wikidata': 0})
    
        if index_batch * BATCH_SIZE < len(countries_list):
            time.sleep(1.5)
    
    df_wikidata = pd.DataFrame(wikidata_rows)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df_wikidata.to_csv(csv_path, index=False)

df_wikidata


Loading Wikidata GLAM data from cache...


,iso_code,glam_count_wikidata
0,IT,14236
1,DE,9370
2,NL,1539
3,PT,554
4,ES,2813
5,FR,2640


#### Integrating the result
The four disparate tables (`df_countries`, `df_europeana`, `df_exp_filtered`, `df_wikidata`) are merged via an inner relational join on the `iso_code` primary key.
A new column is added with the **GLAM Partecipation Rate (%)**: 
  $$\text{Partecipation Rate} = \left(\frac{\text{europeana\_providers}}{\text{glam\_count\_wikidata}}\right) \times 100$$
**Display Formatting**: Numeric variables are cast to float/int and formatted using Pandas Styler (`{:,.0f}` for totals, `{:.2f}%` for rates) to ensure clean tabular presentation without modifying underlying float values.

In [12]:
# Data integration
df_final = df_countries.merge(df_europeana, on='iso_code') \
                    .merge(df_exp_filtered, on='iso_code') \
                    .merge(df_wikidata, on='iso_code') 

df_final['partecipation_rate_glam'] = (df_final['europeana_providers'] / df_final['glam_count_wikidata']) * 100

formatted_df = df_final.style.format({
    'europeana_total': '{:,.0f}',
    'partecipation_rate_glam': '{:.2f}%',
    'culture_expenditure_gpd_2022': '{:.2f}%'
})

formatted_df

,iso_code,name_en,wd_id,europeana_total,europeana_providers,culture_expenditure_gpd_2022,glam_count_wikidata,partecipation_rate_glam
0,IT,Italy,Q38,"1,832,376",158,0.90%,14236,1.11%
1,DE,Germany,Q183,"8,701,240",375,1.00%,9370,4.00%
2,NL,Netherlands,Q55,"9,204,845",104,1.10%,1539,6.76%
3,PT,Portugal,Q45,"139,858",40,0.90%,554,7.22%
4,ES,Spain,Q29,"6,581,724",275,1.20%,2813,9.78%
5,FR,France,Q142,"4,724,898",50,1.40%,2640,1.89%


#### Visualizing RsQ01
* **Chart Type**: Normalized 100% Stacked Bar Chart (`plotly.express.bar`).
* **Data Transformation**: The partecipation rate is melted into a tidy format representing two complementary percentage states: `Active on Europeana (%)` and `Offline GLAMs (%)`.
* **Visual Encoding**: Countries are sorted in descending order by partecipation rate. Bar text labels display percentages rounded to one decimal place (`textposition='outside'` for active segments, `hoverinfo='none'` to disable interactive tooltips for clean static rendering).

In [13]:
# Visualization Partecipation Rate Glam
df_support = df_final[['iso_code', 'partecipation_rate_glam']].copy()
df_support['Active on Europeana (%)'] = df_support['partecipation_rate_glam']
df_support['Offline GLAMs (%)'] = 100 - df_support['partecipation_rate_glam']

df_support = df_support.sort_values(by='partecipation_rate_glam', ascending=False)

df_tidy = df_support.melt(
    id_vars=['iso_code', 'partecipation_rate_glam'],
    value_vars=['Active on Europeana (%)', 'Offline GLAMs (%)'],
    var_name='Status',
    value_name='Percentage'
)

fig1 = px.bar(
    df_tidy,
    x='iso_code',
    y='Percentage',
    color='Status',
    title="The Partecipation Gap: distribution of active europeana providers",
    labels={
        'iso_code': 'Country',
        'Percentage': 'Share of Physical GLAMs (%)',
        'Status': 'Institutional Status'
    },
    color_discrete_map={
        'Active on Europeana (%)': '#5A5A5A',
        'Offline GLAMs (%)': '#AACAE0'
    }
)

fig1.update_traces(
    texttemplate='',
    hoverinfo='none'
)

df_active = df_tidy[df_tidy['Status'] == 'Active on Europeana (%)']

fig1.add_trace(
    go.Scatter(
        x=df_active['iso_code'],
        y=df_active['Percentage'],
        text=df_active['Percentage'].map('{:.1f}%'.format),
        mode='text',
        textposition='top center',
        textfont=dict(size=12, color='black'),
        showlegend=False,
        hoverinfo='none'
    )
)

fig1.update_layout(
    title_x=0.5,
    template="plotly_white",
    height=600,
    width=800,
    barmode='stack',
    hovermode=False,
    xaxis=dict(showgrid=False),
    yaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%",
        range=[0, 108]
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

fig1.show()

#### Visualizing RsQ02
* **Chart Type**: Scatter Plot with Ordinary Least Squares (OLS) Regression Line.
* **Variables**: X-axis encodes public cultural expenditure (% GDP); Y-axis encodes total Europeana digital objects.
* **Model Parameters**: An OLS trendline ($y = mx + q$) is fitted using `numpy.polyfit`. To test the hypothesis of zero digital output at zero public spending, a forced-origin regression ($q = 0$) is calculated where $m = \frac{\sum(xy)}{\sum(x^2)}$.

In [14]:
# Visualization Culture expenditure correlation
x_vals = df_final['culture_expenditure_gpd_2022'].values
y_vals = df_final['europeana_total'].values

m = np.sum(x_vals * y_vals) / np.sum(x_vals**2)
q = 0

x_line = np.linspace(0, x_vals.max() + 0.05, 100)
y_line = m * x_line

fig2 = px.scatter(
    df_final,
    x="culture_expenditure_gpd_2022",
    y="europeana_total",
    text="iso_code",
    color="iso_code",
    color_discrete_map=COUNTRY_COLORS_ISO,
    title="Relationship between public funding and total shared items in Europeana",
    labels={
        "culture_expenditure_gpd_2022": "Public Funding in Culture (% of GDP)",
        "europeana_total": "Total Objects Shared on Europeana"
    },
    hover_data={
        "europeana_total": ":,.0f",
        "culture_expenditure_gpd_2022": False,
        "iso_code": False
    }
)

fig2.add_trace(
    go.Scatter(
        x=x_line,
        y=y_line,
        mode="lines",
        name="Global Trendline (OLS)",
        line=dict(dash="dash", color="#FF4B4B", width=2.5),
        showlegend=False,
        hoverinfo="skip"
    )
)

fig2.update_layout(
    template="plotly_white",
    height=550,
    width=750,
    showlegend=False,
    xaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%",
        range=[x_vals.min() - 0.08, x_vals.max() + 0.08]
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        tickformat=","
    )
)

fig2.update_traces(
    selector=dict(mode="markers+text"),
    textposition="top center",
    textfont=dict(size=12, family="sans-serif", color="black")
)


fig2.show()

#### Visualizing RsQ03 
* **Chart Type**: Multivariate Bubble Chart (`plotly.express.scatter`).
* **Visual Encoding**: X-axis = GLAM Partecipation Rate (%); Y-axis = Public Cultural Expenditure (% GDP); Bubble Area (`size`) = Total Europeana Objects (`size_max=65`).


In [15]:
# Visualization relationship between investment, glam partecipation rate, europeana total item for each selected countries
fig3 = px.scatter(
    df_final,
    x="partecipation_rate_glam",
    y="culture_expenditure_gpd_2022",
    size="europeana_total",
    color="iso_code",
    color_discrete_map=COUNTRY_COLORS_ISO,
    text="iso_code",
    hover_name="name_en",

    hover_data={
        "europeana_total": ":,",
        "partecipation_rate_glam": ":.2f",
        "culture_expenditure_gpd_2022": ":.2f%",
        "iso_code": False
    },

    title="European(a) Digital Heritage: relationship between investment, <br>glam partecipation rate and total shared items</br>",
    labels={
        "partecipation_rate_glam": "GLAM Partecipation Rate (% of physical institutions active online)",
        "culture_expenditure_gpd_2022": "Public Funding in Culture (% of GDP)",
        "europeana_total": "Total Objects in Europeana",
    },
    size_max=65
)

fig3.update_layout(
    title_x=0.5,
    template="plotly_white",
    height=800,
    width=750,
    showlegend=False,
    margin=dict(t=80),
    xaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%"
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%"
    )
)

fig3.update_traces(
    textposition="top center",
    textfont=dict(size=12, family="sans-serif", color="black")
)

fig3.show()



#### Updating Visualization RsQ03
**Sensitivity Override**: To test metric stability against crowdsourced semantic graph under-sampling, the Wikidata baseline for Portugal (`PT`) is replaced with official administrative census data manually collected from INE, recording the most recent numerical data of  museums, libraries and galleries (`portugal_glam_census.csv`, $N=3,395$). The X-axis spatial grid (`range=[-0.5, 12.5]`) is locked across both charts to visually isolate the mathematical impact of the denominator override.

In [16]:
# Update with data from portugal GLAM census
df_ine = pd.read_csv('data/csv/portugal_glam_census.csv')
pt_real_glam_total = df_ine['Value'].sum()
df_final.loc[df_final['iso_code'] == 'PT', 'glam_count_wikidata'] = pt_real_glam_total

df_final['partecipation_rate_glam'] = (df_final['europeana_providers'] / df_final['glam_count_wikidata']) * 100

fig3 = px.scatter(
    df_final,
    x="partecipation_rate_glam",
    y="culture_expenditure_gpd_2022",
    size="europeana_total",
    color="iso_code",
    color_discrete_map=COUNTRY_COLORS_ISO,
    text="iso_code",
    hover_name="name_en",
    hover_data={
        "europeana_total": ":,",
        "partecipation_rate_glam": ":.2f",
        "culture_expenditure_gpd_2022": ":.2f%",
        "iso_code": False
    },
    title="Updated European(a) Digital Heritage: relationship between investment, <br> glam partecipation rate (updated) and total shared items </br>",
    labels={
        "partecipation_rate_glam": "GLAM Partecipation Rate (% of physical institutions active online)",
        "culture_expenditure_gpd_2022": "Public Funding in Culture (% of GDP)",
        "europeana_total": "Total Objects in Europeana",
    },
    size_max=65
)

fig3.update_layout(
    title_x=0.5,
    template="plotly_white",
    height=800,
    width=750,
    showlegend=False,
    margin=dict(t=80),
    xaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%"
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%"
    )
)

fig3.update_traces(
    textposition="top center",
    textfont=dict(size=12, family="sans-serif", color="black")
)

fig3.show()

## RQ02 Who are the providers contributing content to Europeana, and how are they distributed?

### RsQ01 Which are the providers contributing to Europeana?

### RsQ02 How are these providers geographically distributed across Europe?

### RsQ03

### RsQ04 What types of providers contribute to Europeana?


#### Querying the Europeana API for full provider lists and volume percentages

Data is collected via the Europeana Search REST API. The functions iteratively query each country to retrieve the full list of providers and their respective item volumes.

Based on these calculated volumes, the lists are then sorted and truncated to the top 100 providers per country. This optimizes the subsequent classification process while ensuring the relative weight of the national aggregates is accurately captured.

In [34]:
# Europeana Provider Data Collection
import re

TARGET_COUNTRIES = [
    "italy", "france", "germany", "spain", "netherlands", "portugal",
]
LANG_BY_COUNTRY = {
    "italy": "it", "france": "fr", "germany": "de",
    "spain": "es", "netherlands": "nl", "portugal": "pt",
}

API_KEY = EUROPEANA_API_KEY

def europeana_search(query="*", qf=None, facet=None, rows=0, **extra):
    params = {"wskey": API_KEY, "query": query, "rows": rows, **extra}
    if qf:
        params["qf"] = qf
    if facet:
        params["facet"] = facet
        params["profile"] = "facets"

    for attempt in range(5):
        resp = requests.get(EUROPEANA_SEARCH_URL, params=params)
        if resp.status_code == 429:
            wait = 2 ** attempt
            print(f"Rate limited, waiting {wait}s...")
            time.sleep(wait)
            continue
        if resp.status_code != 200:
            print("URL:", resp.url)
            print("Status:", resp.status_code)
            print("Body:", resp.text[:1000])
            resp.raise_for_status()
        time.sleep(REQUEST_DELAY)
        return resp.json()

    raise RuntimeError(f"Failed after retries: {params}")


test = europeana_search(query="*", qf=["COUNTRY:italy"], rows=1)
print("success:", test.get("success"), "| totalResults:", test.get("totalResults"))

def get_total_items(country):
    result = europeana_search(query="*", qf=[f"COUNTRY:{country}"], rows=0)
    return {"country": country, "total_items": result.get("totalResults", 0)}


totals_df = pd.DataFrame([get_total_items(c) for c in TARGET_COUNTRIES])
display(totals_df)

def get_all_providers(country, page_size=200, max_pages=50):
    rows = []
    offset = 0
    for _ in range(max_pages):
        result = europeana_search(
            query="*",
            qf=[f"COUNTRY:{country}"],
            facet="DATA_PROVIDER",
            rows=0,
            **{"f.DATA_PROVIDER.facet.limit": page_size, "f.DATA_PROVIDER.facet.offset": offset},
        )
        facets = result.get("facets", [])
        page_rows = [
            {"country": country, "provider": f["label"], "count": f["count"]}
            for facet in facets
            for f in facet.get("fields", [])
        ]
        rows.extend(page_rows)
        if len(page_rows) < page_size:
            break
        offset += page_size
    return rows


providers_by_country = {}
target_folder = DATA_DIR / "csv" / "providers_data"
target_folder.mkdir(parents=True, exist_ok=True)

for country in TARGET_COUNTRIES:
    rows = get_all_providers(country)
    country_df = pd.DataFrame(rows)
    country_df.to_csv(target_folder / f"providers_{country}.csv", index=False)
    providers_by_country[country] = country_df
    print(f"{country}: {len(country_df)} distinct providers -> providers_{country}.csv")

providers_df = pd.concat(providers_by_country.values(), ignore_index=True)


success: True | totalResults: 1832376


,country,total_items
0,italy,1832376
1,france,4724898
2,germany,8701240
3,spain,6581724
4,netherlands,9204845
5,portugal,139858


italy: 158 distinct providers -> providers_italy.csv
france: 50 distinct providers -> providers_france.csv
germany: 375 distinct providers -> providers_germany.csv
spain: 275 distinct providers -> providers_spain.csv
netherlands: 104 distinct providers -> providers_netherlands.csv
portugal: 40 distinct providers -> providers_portugal.csv


In [36]:
# what % of total item volume the top N providers represent - how large N needs to be.

def coverage_at_n(country_df, n):
    sorted_df = country_df.sort_values("count", ascending=False)
    return sorted_df["count"].head(n).sum() / sorted_df["count"].sum()

for country, df in providers_by_country.items():
    print(f"{country}: top 20 -> {coverage_at_n(df, 20):.1%}, top 50 -> {coverage_at_n(df, 50):.1%}, "
          f"top 100 -> {coverage_at_n(df, 100):.1%} (of {len(df)} total providers)")

italy: top 20 -> 90.2%, top 50 -> 98.4%, top 100 -> 99.9% (of 158 total providers)
france: top 20 -> 99.7%, top 50 -> 100.0%, top 100 -> 100.0% (of 50 total providers)
germany: top 20 -> 79.6%, top 50 -> 92.2%, top 100 -> 97.7% (of 375 total providers)
spain: top 20 -> 76.7%, top 50 -> 92.3%, top 100 -> 98.4% (of 275 total providers)
netherlands: top 20 -> 89.8%, top 50 -> 99.5%, top 100 -> 100.0% (of 104 total providers)
portugal: top 20 -> 99.7%, top 50 -> 100.0%, top 100 -> 100.0% (of 40 total providers)


In [ ]:
# Aggregating Top Providers per Country
# cut each country's provider list down to its top 100 by item count
TOP_N_PER_COUNTRY = 100

top_providers_rows = []
for country, df in providers_by_country.items():
    top = df.sort_values("count", ascending=False).head(TOP_N_PER_COUNTRY)
    top_providers_rows.append(top)

top_providers_df = pd.concat(top_providers_rows, ignore_index=True)
distinct_providers = top_providers_df[["provider", "country"]].drop_duplicates()
print(f"{len(top_providers_df)} (country, provider) rows / {len(distinct_providers)} distinct providers selected")


490 (country, provider) rows / 490 distinct providers selected


#### Querying the remote Wikidata APIs for provider metadata

To later classify the providers into specific typologies, both the Wikidata REST API and the SPARQL endpoint (`https://query.wikidata.org/sparql`) are queried.

* **REST API Entity Resolution**: The provider's name is searched (first in English, falling back to local language) to retrieve the entity's direct description.
* **SPARQL Endpoint Properties**: The endpoint is queried to extract two specific structural properties:
  * `wdt:P31` (instance of): Indicates the entity's class (e.g., museum, national library).
  * `wdt:P101` (field of work): Defines the entity's specialization (e.g., archaeology, broadcasting).
* **Batch Processing**: Provider QIDs are batched in groups of 50 (`batch_size=50`) to optimize query execution and prevent endpoint timeouts.
* **Output**: Returns the raw text descriptions, `instance_of`, and `field_of_work` properties, which form the basis for the subsequent classification.

In [ ]:
# Wikidata Metadata Collection
WD_SEARCH_URL = "https://www.wikidata.org/w/api.php"
WD_HEADERS = {
    "User-Agent": "InfoVis-course-project/0.1 (student project; contact: your.email@example.com)"
}
WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"


def search_wikidata_rest(name, lang="en", max_retries=3):
    params = {
        "action": "wbsearchentities", "search": name, "language": lang,
        "format": "json", "limit": 1, "type": "item",
    }
    for attempt in range(max_retries):
        try:
            resp = requests.get(WD_SEARCH_URL, params=params, headers=WD_HEADERS, timeout=10)
        except requests.exceptions.RequestException as e:
            print(f"network error searching '{name}': {e}")
            time.sleep(2 ** attempt)
            continue
        if resp.status_code == 200:
            results = resp.json().get("search", [])
            if not results:
                return {"qid": None, "description": None, "desc_lang": None, "status": "no_match"}
            top = results[0]
            display_desc = top.get("display", {}).get("description", {})
            description = display_desc.get("value") or top.get("description")
            actual_lang = display_desc.get("language")
            return {"qid": top["id"], "description": description,
                    "desc_lang": actual_lang if description else None, "status": "ok"}
        if resp.status_code in (429, 502, 503):
            time.sleep(2 ** attempt)
            continue
        return {"qid": None, "description": None, "desc_lang": None, "status": f"http_{resp.status_code}"}
    return {"qid": None, "description": None, "desc_lang": None, "status": "retries_exhausted"}


def resolve_provider_wikidata(name, country):
    result = search_wikidata_rest(name, lang="en")
    if result["status"] == "ok" and result["description"]:
        result["resolution_lang"] = "en"
        return result
    country_lang = LANG_BY_COUNTRY.get(country, "en")
    local_result = search_wikidata_rest(name, lang=country_lang)
    if local_result["status"] == "ok" and (local_result["description"] or not result["qid"]):
        local_result["resolution_lang"] = country_lang
        return local_result
    if result["status"] == "ok":
        result["resolution_lang"] = "en"
        return result
    local_result["resolution_lang"] = country_lang
    return local_result


WD_RESOLUTION_FILE = DATA_DIR / "json" / "wikidata_resolution.json"
if WD_RESOLUTION_FILE.exists():
    with open(WD_RESOLUTION_FILE, "r", encoding="utf-8") as f:
        wikidata_resolution = json.load(f)
    print("Loaded wikidata resolution from cache.")
else:
    wikidata_resolution = {}
    for _, row in distinct_providers.iterrows():
        name, country = row["provider"], row["country"]
        result = resolve_provider_wikidata(name, country)
        wikidata_resolution[name] = result
        time.sleep(0.3)
    with open(WD_RESOLUTION_FILE, "w", encoding="utf-8") as f:
        json.dump(wikidata_resolution, f, indent=2, ensure_ascii=False)
    print("Saved wikidata resolution to cache.")
wd_resolved = sum(1 for r in wikidata_resolution.values() if r["status"] == "ok")
print(f"{wd_resolved} / {len(wikidata_resolution)} resolved via Wikidata")

def fetch_institution_info(qids, batch_size=50):
    results = {}
    qids = [q for q in qids if q]
    for i in range(0, len(qids), batch_size):
        batch = qids[i:i + batch_size]
        values = " ".join(f"wd:{q}" for q in batch)
        query = f"""
        SELECT ?item ?instanceOfLabel ?fieldOfWorkLabel WHERE {{
          VALUES ?item {{ {values} }}
          OPTIONAL {{ ?item wdt:P31 ?instanceOf . ?instanceOf rdfs:label ?instanceOfLabel . FILTER(LANG(?instanceOfLabel) = "en") }}
          OPTIONAL {{ ?item wdt:P101 ?fieldOfWork . ?fieldOfWork rdfs:label ?fieldOfWorkLabel . FILTER(LANG(?fieldOfWorkLabel) = "en") }}
        }}
        """
        resp = requests.get(WIKIDATA_SPARQL, params={"query": query, "format": "json"}, headers=WD_HEADERS)
        if resp.status_code != 200:
            continue
        for row in resp.json()["results"]["bindings"]:
            qid = row["item"]["value"].rsplit("/", 1)[-1]
            entry = results.setdefault(qid, {"instance_of": [], "field_of_work": []})
            if "instanceOfLabel" in row:
                entry["instance_of"].append(row["instanceOfLabel"]["value"])
            if "fieldOfWorkLabel" in row:
                entry["field_of_work"].append(row["fieldOfWorkLabel"]["value"])
        time.sleep(0.5)
    return results


SPARQL_INFO_FILE = DATA_DIR / "json" / "sparql_info.json"
if SPARQL_INFO_FILE.exists():
    with open(SPARQL_INFO_FILE, "r", encoding="utf-8") as f:
        sparql_info = json.load(f)
    print("Loaded SPARQL info from cache.")
else:
    qid_list = [info["qid"] for info in wikidata_resolution.values() if info.get("qid")]
    sparql_info = fetch_institution_info(qid_list)
    with open(SPARQL_INFO_FILE, "w", encoding="utf-8") as f:
        json.dump(sparql_info, f, indent=2, ensure_ascii=False)
    print("Saved SPARQL info to cache.")
qid_list = [info["qid"] for info in wikidata_resolution.values() if info.get("qid")]
print(f"instance_of/field_of_work found for {len(sparql_info)} / {len(qid_list)} QIDs")

Saved wikidata resolution to cache.
185 / 490 resolved via Wikidata
Saved SPARQL info to cache.
instance_of/field_of_work found for 185 / 185 QIDs


#### Classifying provider typologies based on Wikidata metadata

To assign each provider to a standardized typology, a cascading rule-based classification function is applied to the metadata previously retrieved.

* **Typology Categories**: Providers are mapped to one of 8 distinct categories (e.g., `library/archive`, `art/history museum`, `audiovisual/film archive`) or marked as `other`.
* **Regex Rule Matching**: A predefined set of regular expressions (`INSTITUTION_RULES`) scans the textual metadata for specific English and localized keywords (e.g., `museum`, `biblioteca`, `university`).
* **Classification Hierarchy**: The algorithm evaluates the metadata in a specific order of precedence to resolve conflicts:
  1. The raw provider name provided by Europeana.
  2. The direct entity description retrieved via the Wikidata REST API.
  3. The structural properties (`instance_of` and `field_of_work`) retrieved via the SPARQL endpoint.
* **Manual Overrides**: For massive or highly ambiguous providers where automated entity resolution fails or returns conflicting data, a hardcoded dictionary (`CONFIRMED_FIXES`) forces the correct categorization.

In [26]:
# Provider Typologies Rules and Classification
INSTITUTION_RULES = [
    (r"film archive|cin[ée]math[èe]que|kinemathek|audiovisual archive|\baudiovisual\b|\bcinema\b|cinecitt[aà]|"
     r"moving image|sound\s*(&|and)\s*vision|film museum|film institute", "audiovisual/film archive"),
    (r"\bmusic\b|musical|ethnomusicology|conservator(y|io|oire)|philharmonic|philharmonie|concert hall",
     "audiovisual/film archive"),

    (r"national library|public library|research library|\blibrar", "library/archive"),
    (r"\barchiv|\barquiv|national archives|repositor", "library/archive"),
    (r"bibliotec|bibliothek", "library/archive"),

    (r"natural history museum|science museum|herbarium|botanic|\bzoo\b|planetarium|aquarium|"
     r"geological survey|tropical.{0,15}research", "natural history/science institution"),

    (r"art museum|history museum|national museum|encyclopedic museum|archaeological museum|"
     r"specialized museum|military museum|open-air museum|\bmuseum\b|\bmuseo\b|mus[ée]e|\bmuseu\b|"
     r"museen|\bgallery\b|art collection|photograph|\bcastle\b|\bpalace\b|historical monument|"
     r"epigraphic|archaeolog|archeolog", "art/history museum"),

    (r"broadcaster|television|radio station|newspaper|press agency|publishing house",
     "media/broadcast organization"),

    (r"\buniversity\b|research institute|academy of sciences|research cent(er|re)|\bresearch\b|"
     r"\blaboratory\b|laboratoire", "academic/research institution"),

    (r"government agency|ministry|municipality|public administration|state institution|\bgovernment\b|"
     r"gobierno|ajuntament|gemeente|province of|heritage agency|superintendence|\bcourt\b",
     "government/administrative body"),
]
_compiled_institution_rules = [(re.compile(pat, re.IGNORECASE), name) for pat, name in INSTITUTION_RULES]


def classify_text(text):
    if not text:
        return None
    for pattern, category in _compiled_institution_rules:
        if pattern.search(text):
            return category
    return "other"


def classify_from_sparql(qid):
    info = sparql_info.get(qid, {"instance_of": [], "field_of_work": []})

    fow_category = classify_text(" | ".join(info["field_of_work"]))
    if fow_category not in (None, "other"):
        return fow_category

    io_category = classify_text(" | ".join(info["instance_of"]))
    if io_category not in (None, "other"):
        return io_category

    if not info["field_of_work"] and not info["instance_of"]:
        return None
    return "other"


def classify_provider(name):
    name_category = classify_text(name)
    wd_info = wikidata_resolution.get(name)
    desc_category = classify_text(wd_info.get("description")) if wd_info else None

    if name_category not in (None, "other") and desc_category not in (None, "other") and name_category != desc_category:
        # disagreement -- flag for review rather than silently picking one
        return "other", "name_description_conflict"
    if name_category not in (None, "other"):
        return name_category, "provider_name"
    if desc_category not in (None, "other"):
        return desc_category, "wikidata_description"

    if wd_info:
        sparql_category = classify_from_sparql(wd_info.get("qid"))
        if sparql_category not in (None, "other"):
            return sparql_category, "wikidata_sparql_fallback"
        if desc_category == "other" or sparql_category == "other" or name_category == "other":
            return "other", "unmatched"

    if name_category == "other":
        return "other", "unmatched"

    return None, None

classification_rows = []
for name in distinct_providers["provider"]:
    category, source = classify_provider(name)
    classification_rows.append({"provider": name, "provider_category": category, "classification_source": source})

classification_df = pd.DataFrame(classification_rows)
print(classification_df["provider_category"].value_counts(dropna=False))
print()
print(classification_df["classification_source"].value_counts(dropna=False))


provider_category
library/archive                        201
art/history museum                     139
other                                   78
academic/research institution           29
audiovisual/film archive                17
government/administrative body          14
natural history/science institution      9
media/broadcast organization             3
Name: count, dtype: int64

classification_source
provider_name                373
unmatched                     71
wikidata_description          29
wikidata_sparql_fallback      10
name_description_conflict      7
Name: count, dtype: int64


In [39]:
# Provider Classification and Manual Override
CONFIRMED_FIXES = {
    # Entity-resolution errors (Wikidata matched the wrong entity or a misleading description)
    "Brixiana": "library/archive",
    "Paul Van Riel": "other",

    # Government heritage-protection agencies mismatched by museum/archaeology keywords
    "Cultural Heritage Agency of the Netherlands": "government/administrative body",
    "Historical Monuments: Regional Conservation": "government/administrative body",
    "Ministry of Culture and Communication, Regional Archaeology Service": "government/administrative body",

    # Archaeology/epigraphic RESEARCH institutes mismatched as museums (the "archaeolog"/"epigraphic"
    "German Archaeological Institute": "academic/research institution",
    "Epigraphic Database Roma": "academic/research institution",
    "Epigraphic Dabatase Bari": "academic/research institution",
    "University Institute for Research in Iberian Archeology": "academic/research institution",
    "CISA -Interdipartimental Center for Archaeology": "academic/research institution",

    # "Conservatory" false-friend matches
    "National Conservatory of Arts and Crafts": "academic/research institution",
    "Conservatory of the Gironde Estuary": "government/administrative body",

    # Science/technology museums swept into the generic art/history museum bucket
    "Museon-Omniversum": "natural history/science institution",
    "Technoseum": "natural history/science institution",
    "Zoological Research Museum Koenig": "natural history/science institution",
}

review_rows = []
for name in distinct_providers["provider"]:
    if name in CONFIRMED_FIXES:
        continue

    category, source = classify_provider(name)
    if category in (None, "other"):
        wd = wikidata_resolution.get(name, {})
        item_count = top_providers_df.loc[top_providers_df["provider"] == name, "count"].sum()
        review_rows.append({
            "provider": name,
            "description_en": wd.get("description_en"),
            "category": category,
            "source": source,
            "count": item_count,
        })

new_review_df = pd.DataFrame(review_rows).sort_values("count", ascending=False)

review_path = DATA_DIR / "csv" / "manual_review.csv"
if review_path.exists():
    existing_df = pd.read_csv(review_path)
    existing_categories = dict(zip(existing_df["provider"], existing_df["manual_category"]))
    new_review_df["manual_category"] = new_review_df["provider"].map(existing_categories).fillna("")
else:
    new_review_df["manual_category"] = ""

new_review_df.to_csv(review_path, index=False)
print(f"{len(new_review_df)} providers needed review, {new_review_df['count'].sum():,} total items")
filled = (new_review_df["manual_category"] != "").sum()

review_df = pd.read_csv(DATA_DIR / "csv" / "manual_review.csv")
review_df["manual_category"] = review_df["manual_category"].fillna("")
manual_overrides = {
    row["provider"]: row["manual_category"].strip()
    for _, row in review_df.iterrows()
    if row["manual_category"].strip()
}
print(f"{len(manual_overrides)} manual overrides loaded from CSV")
print(f"{len(CONFIRMED_FIXES)} confirmed fixes hard-coded")

def classify_provider_final(name):
    if name in CONFIRMED_FIXES:
        return CONFIRMED_FIXES[name], "confirmed_fix"
    if name in manual_overrides:
        return manual_overrides[name], "manual_override"
    return classify_provider(name)

final_categories = {name: classify_provider_final(name)[0] for name in distinct_providers["provider"]}

top_providers_df["provider_category"] = top_providers_df["provider"].map(final_categories)
top_providers_df["provider_category"] = top_providers_df["provider_category"].fillna("unresolved")

category_by_country = (
    top_providers_df.groupby(["country", "provider_category"])["count"]
    .sum()
    .unstack(fill_value=0)
)

category_by_country = category_by_country.merge(
    totals_df.set_index("country")["total_items"], left_index=True, right_index=True
)

sample_total = category_by_country.drop(columns="total_items").sum(axis=1)

category_share = category_by_country.drop(columns="total_items").div(
    category_by_country["total_items"], axis=0
)

category_share["coverage"] = sample_total / category_by_country["total_items"]

category_share = (category_share * 100).round(2)
category_share = category_share.reset_index()

category_share

audit_rows = []
for name in distinct_providers["provider"]:
    category, source = classify_provider_final(name)
    count = top_providers_df.loc[top_providers_df["provider"] == name, "count"].sum()
    country = distinct_providers.loc[distinct_providers["provider"] == name, "country"].iloc[0]
    audit_rows.append({
        "provider": name,
        "country": country,
        "count": count,
        "provider_category": category if category else "unresolved",
        "classification_source": source if source else "unresolved",
    })

cat_df = pd.DataFrame(audit_rows).sort_values(["country", "count"], ascending=[True, False])
cat_df.to_csv(DATA_DIR / "csv" / "providers_classified_complete.csv", index=False)

pd.set_option("display.max_rows", None)
cat_df.head(30)


77 providers needed review, 1,834,363 total items
77 manual overrides loaded from CSV
15 confirmed fixes hard-coded


,provider,country,count,provider_category,classification_source
100,National Library of France,france,2999310,library/archive,provider_name
101,Media Library of Architecture and Heritage,france,510877,library/archive,provider_name
102,Natural History Museum in Paris,france,502160,natural history/science institution,provider_name
103,Ministry of Culture,france,173279,government/administrative body,provider_name
104,Historical Monuments: Regional Conservation,france,128192,government/administrative body,confirmed_fix
105,"Ministry of Culture and Communication, Regiona...",france,110142,government/administrative body,confirmed_fix
106,National Audiovisual Institute France,france,54491,audiovisual/film archive,provider_name
107,Interuniversity Health Library,france,47992,library/archive,provider_name
108,Palais Galliera - Musée de la Mode de la Ville...,france,44495,art/history museum,provider_name
109,Mobilier National Collections,france,24284,art/history museum,manual_override


#### Visualizing RsQ01
To answer RsQ01, an interactive visualization is generated to explore the data at the national level.

* **Top Providers by Country (Horizontal Bar Chart)**
  * **Chart Type**: Interactive Horizontal Bar Chart (`plotly.graph_objects.Bar`).
  * **Visual Encoding**: Displays the top 15 providers for a selected country. The x-axis represents the percentage of that country's total items contributed by each provider. 
* **Interactivity**: A dropdown menu allows dynamically switching between the 6 target countries.

In [51]:
# Visualization Top Providers typologies per country
TOP_N_DISPLAY = 15

provider_bar_state = {}
for country in totals_df["country"]:
    country_total = totals_df.loc[totals_df["country"] == country, "total_items"].iloc[0]
    top20 = (
        top_providers_df[top_providers_df["country"] == country]
        .sort_values("count", ascending=False)
        .head(TOP_N_DISPLAY)
        .copy()
    )
    top20["pct"] = top20["count"] / country_total * 100
    top20 = top20.sort_values("pct", ascending=True)  # ascending so the largest ends up at the TOP of the horizontal bar

    provider_bar_state[country] = {
        "labels": list(top20["provider"]),
        "values": list(top20["pct"]),
        "hover": [
            f"<b>{p}</b><br>{c:,} items<br>{pct:.1f}% of {country.capitalize()}\'s total"
            for p, c, pct in zip(top20["provider"], top20["count"], top20["pct"])
        ],
    }

default_country = totals_df["country"].iloc[0]
default_state = provider_bar_state[default_country]

fig_bar = go.Figure(go.Bar(
    x=default_state["values"],
    y=default_state["labels"],
    orientation="h",
    marker=dict(color=COUNTRY_COLORS.get(default_country, "#CCCCCC")),
    customdata=default_state["hover"],
    hovertemplate="%{customdata}<extra></extra>",
    text=[f"{v:.1f}%" for v in default_state["values"]],
    textposition="outside",
    textfont=dict(color="black"),
    cliponaxis=False,
))

buttons = []
for country in totals_df["country"]:
    state = provider_bar_state[country]
    buttons.append(dict(
        label=country.capitalize(),
        method="update",
        args=[
            {
                "x": [state["values"]],
                "y": [state["labels"]],
                "customdata": [state["hover"]],
                "text": [[f"{v:.1f}%" for v in state["values"]]],
                "marker.color": [COUNTRY_COLORS.get(country, "#CCCCCC")],
            },
            {"title.text": f"Top {TOP_N_DISPLAY} providers -- {country.capitalize()}"},
        ],
    ))

fig_bar.update_layout(
    title=dict(text=f"Top {TOP_N_DISPLAY} providers -- {default_country.capitalize()}", x=0.5, xanchor="center"),
    xaxis_title="% of country's total items",
    height=700,
    margin=dict(l=250, r=60, t=120, b=40),  # generous left margin for long provider names
    updatemenus=[dict(
        type="dropdown", direction="down", x=1.0, y=1.15, xanchor="right",
        buttons=buttons, showactive=True,
    )],
    paper_bgcolor="white",
    plot_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="#EEEEEE", zeroline=False),
    yaxis=dict(showgrid=False),
)

fig_bar.show()


#### Visualizing RsQ04
To answer RsQ04 (provider typologies), two interactive visualizations are generated to explore the data at both the aggregate and national levels.

* **Overall Typology Distribution (Pie Chart)**
  * **Chart Type**: Pie Chart (`plotly.graph_objects.Pie`).
  * **Visual Encoding**: Aggregates the item volumes across all 6 countries to show the macro-level distribution of provider typologies. Categories are color-coded consistently using the shared `CATEGORY_COLORS` palette.

* **Typology Contribution by Country (Sunburst Chart)**
  * **Chart Type**: Multi-level Sunburst Chart (`plotly.graph_objects.Sunburst`).
  * **Visual Encoding**: The inner ring represents the proportional contribution of each country to the total item volume of the 6-country dataset. The outer ring breaks down each country's slice into its specific provider typologies.

In [29]:
# Visualization Category share within and across countries
# (a) category share WITHIN each country -- against that country's true total item count
category_by_country = cat_df.groupby(["country", "provider_category"])["count"].sum().unstack(fill_value=0)
category_by_country = category_by_country.merge(
    totals_df.set_index("country")["total_items"], left_index=True, right_index=True
)

sample_total = category_by_country.drop(columns="total_items").sum(axis=1)

category_share_by_country = category_by_country.drop(columns="total_items").div(
    category_by_country["total_items"], axis=0
)
category_share_by_country["coverage"] = sample_total / category_by_country["total_items"]
category_share_by_country = (category_share_by_country * 100).round(1).reset_index()

category_share_by_country.to_csv(DATA_DIR / "csv" / "category_share_by_country.csv", index=False)
print("Category share within each country (%):")
category_share_by_country

# (b) each category's share of the OVERALL six-country total -- "contribution to Europeana" in aggregate
overall_total = cat_df["count"].sum()

overall_category_share = (
    cat_df.groupby("provider_category")["count"]
    .sum()
    .reset_index()
    .rename(columns={"count": "items"})
)
overall_category_share["pct_of_europeana_total"] = (overall_category_share["items"] / overall_total * 100).round(1)
overall_category_share = overall_category_share.sort_values("pct_of_europeana_total", ascending=False)

overall_category_share.to_csv(DATA_DIR / "csv" / "category_share_overall.csv", index=False)
print(f"\nCategory share of the full six-country total ({overall_total:,} items):")
overall_category_share

plot_df = overall_category_share.sort_values("pct_of_europeana_total", ascending=False)

fig_overall = go.Figure(go.Pie(
    labels=plot_df["provider_category"],
    values=plot_df["items"],
    marker=dict(colors=[CATEGORY_COLORS.get(c, "#D1D5DB") for c in plot_df["provider_category"]]),
    textinfo="none",
    textfont=dict(color="black"),
    insidetextorientation="horizontal",
    hovertemplate="<b>%{label}</b><br>%{value:,} items<br>%{percent} of the six-country total<extra></extra>",
))

fig_overall.update_layout(
    title=dict(text="Category share of Europeana's total items (six countries combined)", x=0.5, xanchor="center"),
    height=600,
    margin=dict(l=40, r=40, t=90, b=40),
    showlegend=True,
    legend=dict(title="Category"),
    paper_bgcolor="white",
    plot_bgcolor="white",
    uniformtext=dict(minsize=10, mode="hide"),
)

fig_overall.show()


Category share within each country (%):

Category share of the full six-country total (30,879,627 items):


In [30]:
# Visualization Country contribution to Europeana, by provider category
COUNTRY_COLORS = {
    "netherlands": "#a180ad",
    "france": "#1f7f95",
    "portugal": "#f4a64e",
    "italy": "#90BE6D",
    "germany": "#feda15",
    "spain": "#bb521f",
}

CATEGORY_COLORS = {
    "audiovisual/film archive": "#80CBC4",
    "art/history museum": "#9FA8DA",
    "natural history/science institution": "#CE93D8",
    "library/archive": "#90CAF9",
    "academic/research institution": "#FFCC80",
    "media/broadcast organization": "#EF9A9A",
    "government/administrative body": "#E8A0BE",
    "other": "#CFCFCF",
    "unresolved": "#9E9E9E",
}

provider_counts_by_country = distinct_providers.groupby("country").size()
total_providers = len(distinct_providers)

sunburst_country = cat_df.groupby("country")["count"].sum().reset_index()
sunburst_cat = cat_df.groupby(["country", "provider_category"])["count"].sum().reset_index()

ids, labels, parents, values, colors, hover_text, text_sizes = [], [], [], [], [], [], []

for _, row in sunburst_country.iterrows():
    country = row["country"]
    item_count = row["count"]
    n_providers = provider_counts_by_country.get(country, 0)
    provider_pct = n_providers / total_providers * 100

    ids.append(country)
    labels.append(country.capitalize())
    parents.append("")
    values.append(item_count)
    colors.append(COUNTRY_COLORS.get(country, "#CCCCCC"))
    hover_text.append(
        f"<b>{country.capitalize()}</b><br>{item_count:,} items<br>{provider_pct:.1f}% of all providers"
    )
    text_sizes.append(22)  # country ring -- larger

for _, row in sunburst_cat.iterrows():
    country, cat, count = row["country"], row["provider_category"], row["count"]
    country_total = sunburst_country.loc[sunburst_country["country"] == country, "count"].iloc[0]
    cat_pct_of_country = count / country_total * 100

    ids.append(f"{country}-{cat}")
    labels.append(cat)
    parents.append(country)
    values.append(count)
    colors.append(CATEGORY_COLORS.get(cat, "#D1D5DB"))
    hover_text.append(
        f"<b>{cat}</b><br>{count:,} items<br>{cat_pct_of_country:.1f}% of {country.capitalize()}"
    )
    text_sizes.append(14)  # category ring -- normal

fig = go.Figure(go.Sunburst(
    ids=ids,
    labels=labels,
    parents=parents,
    values=values,
    branchvalues="total",
    marker=dict(colors=colors),
    customdata=hover_text,
    hovertemplate="%{customdata}<extra></extra>",
    insidetextorientation="horizontal",
    textfont=dict(color="black", size=text_sizes),
    insidetextfont=dict(color="black", size=text_sizes),
))

categories_present = sunburst_cat["provider_category"].unique()
for cat in categories_present:
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode="markers",
        marker=dict(size=10, color=CATEGORY_COLORS.get(cat, "#D1D5DB")),
        name=cat,
        showlegend=True,
        hoverinfo="skip",
    ))

fig.update_layout(
    title=dict(text="Country contribution to Europeana, by provider category", x=0.5, xanchor="center"),
    height=750,
    margin=dict(t=80, l=0, r=160, b=0),
    showlegend=True,
    legend=dict(title="Category", yanchor="middle", y=0.5, xanchor="left", x=1.02),
    uniformtext=dict(minsize=7, mode="hide"),
    paper_bgcolor="white",
    plot_bgcolor="white",
    xaxis=dict(visible=False, showgrid=False, zeroline=False),
    yaxis=dict(visible=False, showgrid=False, zeroline=False),
)

fig.show()

## RQ03 To what extent do countries meet Europeana's quality standards and how open are their datasets?